# EDA — NYC Yellow Taxi Trip Records 2023

**Objectifs** : explorer la structure des données, la variable cible `tip_pct`, les distributions, la temporalité, l’activité par zone, les corrélations et les outliers.

**Données** : fichier Parquet TLC si présent, sinon jeu synthétique (`generate_taxi_df`).


In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

ROOT = Path("..").resolve()

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

from src.optim.benchmarks import generate_taxi_df
from src.optim.memory import optimize_dtypes

sns.set_theme(style="darkgrid", palette="muted")


def style_plotly(fig):
    """Thème sombre homogène pour tous les graphiques Plotly."""
    fig.update_layout(title_font_size=16, template="plotly_dark")
    return fig


## Chargement des données

Tentative de lecture du Parquet TLC ; sinon génération de **500 000** lignes synthétiques.

In [ ]:
PARQUET = ROOT / "data" / "raw" / "yellow_tripdata_2023-01.parquet"

if PARQUET.exists():
    df = pd.read_parquet(PARQUET, engine="pyarrow")
    print(f"Chargé depuis : {PARQUET}")
else:
    df = generate_taxi_df(n_rows=500_000)
    print("Parquet absent — jeu synthétique 500k lignes.")

df = optimize_dtypes(df.copy(), verbose=False)
print(df.info())
display(df.describe(include="all").T.head(20))


## Distribution de la variable cible `tip_pct`

`t = tip / fare` (fare plancher 0,01) ; on restreint à \[0, 1\] pour l’histogramme.

In [ ]:
df["tip_pct"] = df["tip_amount"] / df["fare_amount"].clip(lower=0.01)
tip_df = df.loc[(df["tip_pct"] >= 0) & (df["tip_pct"] <= 1), ["tip_pct"]].copy()
mean_tip = float(tip_df["tip_pct"].mean())

fig = px.histogram(
    tip_df,
    x="tip_pct",
    nbins=80,
    title="Distribution du taux de pourboire (tip_pct)",
)
fig.add_vline(
    x=mean_tip,
    line_dash="dash",
    line_color="gold",
    annotation_text=f"Moyenne = {mean_tip:.3f}",
    annotation_position="top",
)
style_plotly(fig)
fig.show()


## Distributions des variables numériques

Six histogrammes (grille 2×3) pour les grandeurs d’usage courant.

In [ ]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60.0

cols = [
    "fare_amount",
    "trip_distance",
    "tip_amount",
    "passenger_count",
    "pickup_hour",
    "trip_duration_min",
]
fig = make_subplots(rows=2, cols=3, subplot_titles=cols)
for i, c in enumerate(cols, start=1):
    row, col = (i - 1) // 3 + 1, (i - 1) % 3 + 1
    s = df[c].dropna()
    fig.add_trace(go.Histogram(x=s, name=c, showlegend=False), row=row, col=col)
fig.update_layout(title_text="Distribution des variables numériques", height=700)
style_plotly(fig)
fig.show()


## Activité horaire

Nombre de courses par **heure de la journée** (agrégation sur tout l’échantillon) ; heures de rush en surbrillance.

In [ ]:
h = df["pickup_hour"].value_counts().sort_index()
# Rush 7-9h et 17-19h : masque booleen (pas de .get sur ndarray)
is_rush = ((h.index >= 7) & (h.index <= 9)) | ((h.index >= 17) & (h.index <= 19))
colors = ["#ff7f0e" if bool(r) else "#1f77b4" for r in is_rush]

fig = go.Figure(
    go.Bar(x=h.index.astype(int), y=h.values, marker_color=colors, name="courses")
)
fig.update_layout(
    title="Activité horaire des taxis NYC",
    xaxis_title="Heure",
    yaxis_title="Nombre de courses",
)
style_plotly(fig)
fig.show()


## Heatmap zone × heure

**Top 20** zones par volume ; intensité = nombre de courses.

In [ ]:
zone_vol = df.groupby("PULocationID").size().sort_values(ascending=False).head(20).index
sub = df[df["PULocationID"].isin(zone_vol)].copy()
sub["hour"] = sub["tpep_pickup_datetime"].dt.hour
pivot = sub.pivot_table(index="PULocationID", columns="hour", values="fare_amount", aggfunc="count", fill_value=0)

fig = px.imshow(
    pivot,
    labels=dict(x="Heure", y="PULocationID", color="Courses"),
    title="Activité par zone et par heure",
    color_continuous_scale="Viridis",
    aspect="auto",
)
style_plotly(fig)
fig.show()


## Matrice de corrélation (Seaborn)

Sous-échantillon pour rester fluide sur de gros jeux.

In [ ]:
sample = df.sample(n=min(50_000, len(df)), random_state=42)
cols_c = ["fare_amount", "tip_pct", "trip_distance", "passenger_count", "pickup_hour"]
mat = sample[cols_c].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(mat, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()


## Outliers : avant / après filtrage

Filtre type TP : `fare_amount` \[2,5 ; 200\], `trip_distance` \[0,1 ; 100\].

In [ ]:
def box_pair(data, title):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("fare_amount", "trip_distance"))
    for j, c in enumerate(["fare_amount", "trip_distance"], start=1):
        fig.add_trace(go.Box(y=data[c], name=c, showlegend=False), row=1, col=j)
    fig.update_layout(title_text=title, height=400)
    return style_plotly(fig)

clean = df[
    (df["fare_amount"] >= 2.5)
    & (df["fare_amount"] <= 200)
    & (df["trip_distance"] >= 0.1)
    & (df["trip_distance"] <= 100)
].copy()

box_pair(df, "Détection et traitement des outliers — brut").show()
box_pair(clean, "Détection et traitement des outliers — filtré").show()


## Top 20 zones par `tip_pct` moyen

Moyenne par `PULocationID` (zones avec assez de volume).

In [ ]:
tip_ok = df.loc[(df["tip_pct"] >= 0) & (df["tip_pct"] <= 1)].copy()
zone_tip = (
    tip_ok.groupby("PULocationID")["tip_pct"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean", ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    zone_tip,
    x="mean",
    y="PULocationID",
    orientation="h",
    title="Top 20 zones par taux de pourboire moyen",
    labels={"mean": "tip_pct moyen", "PULocationID": "Zone"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
style_plotly(fig)
fig.show()


## Conclusions EDA

- **`tip_pct`** : distribution asymétrique avec masse vers 0–0,2 ; corrélation modérée avec `fare_amount` / `trip_distance` selon l’échantillon.
- **Temporalité** : pics matin/soir compatibles avec des rush hours ; utile pour `pickup_hour`, flags rush/week-end en feature engineering.
- **Zones** : forte hétérogénéité du `tip_pct` moyen — justifie encodage cible ou embeddings plutôt qu’un one-hot brut de 265 modalités.
- **Outliers** : queues épaisses sur `fare_amount` et `trip_distance` — filtrage ou winsorisation recommandés en préparation.
- **Suite** : enrichir avec features temporelles (sin/cos), agrégations multi-index, puis modèle supervisé (Partie 4).
